In [42]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [43]:
df = pd.read_csv('train.txt' , sep=';' , header=None , names=['text' , 'emotion'])

In [44]:
df

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger
...,...,...
15995,i just had a very brief time in the beanbag an...,sadness
15996,i am now turning and i feel pathetic that i am...,sadness
15997,i feel strong and good overall,joy
15998,i feel like this was such a rude comment and i...,anger


In [45]:
df.isna().sum()

text       0
emotion    0
dtype: int64

In [46]:
unique_emotions = df['emotion'].unique()
emotion_numbers = {}
i = 0
for emo in unique_emotions:
  emotion_numbers[emo] = i
  i +=1

df['emotion'] = df['emotion'].map(emotion_numbers)

In [47]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [48]:
df['text'] = df['text'].apply(lambda x : x.lower())

In [49]:
df.head(2)

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0


In [50]:
import string

def remove_punc(txt):
    return txt.translate(str.maketrans('','',string.punctuation))

In [51]:
df['text'] = df['text'].apply(remove_punc)

In [52]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)

In [53]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

In [54]:
import nltk

In [55]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [56]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rohan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rohan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [57]:
stop_words = set(stopwords.words('english'))

In [58]:
len(stop_words)

198

In [59]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [60]:
def remove(txt):
    words = word_tokenize(txt)
    cleaned = []
    
    for i in words:
        if not i in stop_words:
            cleaned.append(i)
    
    
    return ' '.join(cleaned)

In [61]:
df['text'] = df['text'].apply(remove)

In [62]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [63]:
from sklearn.model_selection import train_test_split

In [64]:
X_train, X_test, y_train, y_test = train_test_split( df['text'] , df['emotion'] , test_size=0.20, random_state=42)

In [65]:
from sklearn.feature_extraction.text import CountVectorizer , TfidfVectorizer

In [66]:
bow_vectorizer = CountVectorizer()

In [67]:
X_train_bow = bow_vectorizer.fit_transform(X_train)

In [68]:
X_test_bow = bow_vectorizer.fit_transform(X_test)

In [69]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)


nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)


pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))
     

0.7678125


In [71]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)


nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

y_pred = nb2_model.predict(X_test_tfidf)
     

print(accuracy_score(y_test, y_pred))

0.6609375


In [72]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train_tfidf , y_train)

y_predict = model.predict(X_test_tfidf)

print(accuracy_score(y_predict , y_test))

0.8615625


In [73]:
from sklearn.linear_model import LogisticRegression

model2 = LogisticRegression()

model2.fit(X_train_bow , y_train)

y_predict_ = model2.predict(X_test_bow)

print(accuracy_score(y_predict_ , y_test))

0.88875
